# Sustainable Development Index (IDS)

The Sustainable Development Index (IDS) of the SDG Atlas of Bolivia 2026, which summarises 81 indicators of 15 goals, and its national ranking, for every municipality near 2012 and near 2024.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities343-sdg/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Using geo4bolivia, load the Sustainable Development Index (index_sdg) from the dataset sdg and list the 10 municipalities that gained most between ~2012 and ~2024, with their department."
- "Did the municipalities that started furthest behind catch up? Run geo.beta_convergence("sdg", "index_sdg"), plot the result and explain the coefficient in plain words."
- "Make a scatter plot of urbanization against index_sdg in 2024, coloured by department, and tell me whether more urban municipalities score higher."
- "Compute the population-weighted mean of index_sdg by department in both periods (geo.aggregate with dataset="sdg") and rank the departments."
- "Is the IDS spatially clustered? Run geo.moran and geo.lisa on index_sdg in 2024 and describe where the high-high and low-low clusters are."


## Variables

The columns of `sdg`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("sdg")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Period |  |
| `mun_id` | INE municipality code |  |
| `index_sdg` | Sustainable Development Index, IDS (0-100) | index (0-100) |
| `rank_sdg` | IDS national ranking (1 = best) | rank |
| `population` | Population (census) | people |
| `urbanization` | Urbanization rate (% of population) | % of population |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
# The Sustainable Development Index (0-100, higher is better), ~2024
gdf = geo.load("sdg", variables="index_sdg", years=2024, geometry=True)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg": "Sustainable Development Index (0-100)"},
          title="Sustainable Development Index (IDS), ~2024").show()

# Convergence: units that started lower gained more
df = geo.load("sdg", variables="index_sdg")
wide = df.pivot(index="geo_id", columns="year", values="index_sdg")
wide = wide.add_prefix("y")
wide["change"] = wide["y2024"] - wide["y2012"]
wide["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(wide, x="y2012", y="change", hover_name="name",
           labels={"y2012": "IDS, ~2012", "change": "Change in the IDS (points)"},
           title="Initial IDS and its change, ~2012 to ~2024")

## Load the data without the package (plain URLs)

In [ ]:
# No geo4bolivia needed: every file is a plain URL
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
df = pd.read_csv(BASE + "data/sdg/map343/sdg.csv")  # geo_id, year, index_sdg, ...
municipalities = gpd.read_file(BASE + "geo/boundaries/map343.geojson")

ids = df.loc[df["year"] == 2024, ["geo_id", "index_sdg", "rank_sdg"]]
gdf = municipalities.merge(ids, on="geo_id")
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg": "Sustainable Development Index (0-100)"},
          title="Sustainable Development Index (IDS), ~2024")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/sdg/map343/sdg_wide.gpkg",
                    layer="sdg")   # or sdg_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="index_sdg_2024", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"index_sdg_2024": "Sustainable Development Index, IDS (0-100)"},
          title="Sustainable Development Index, IDS (0-100), ~2024")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/sdg/map343/sdg_wide.geojson",
                 column="index_sdg_2024", name="Sustainable Development Index, IDS (0-100), ~2024",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## Which goals moved

In [ ]:
goals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 15, 16, 17]
rows = []
for g in goals:
    df = geo.load(f"sdg{g:02d}", variables=f"index_sdg{g}")
    for year, part in df.groupby("year"):
        rows.append({"goal": f"SDG {g}", "period": f"~{year}",
                     "index": part[f"index_sdg{g}"].mean()})
px.scatter(pd.DataFrame(rows), x="index", y="goal", color="period",
           labels={"index": "Mean goal index (0-100)", "goal": ""},
           title="The 15 goal indices, ~2012 and ~2024")

## Ranking mobility

In [ ]:
df = geo.load("sdg", variables="rank_sdg")
wide = df.pivot(index="geo_id", columns="year", values="rank_sdg")
wide = wide.add_prefix("y")
wide["name"] = geo.units("map343").set_index("geo_id")["name"]
px.scatter(wide, x="y2012", y="y2024", hover_name="name",
           labels={"y2012": "Rank ~2012 (1 = best)", "y2024": "Rank ~2024"},
           title="IDS national ranking, ~2012 and ~2024")

## Rank the departments

In [ ]:
df = geo.load("sdg")
dep = geo.aggregate(df, to="department", variables=["index_sdg"],
                    rules={"index_sdg": "weighted_mean:population"})
dep["department"] = dep["geo_id"].map(
    geo.units("department").set_index("geo_id")["name"])
dep["period"] = "~" + dep["year"].astype(str)
px.scatter(dep.sort_values("index_sdg"), x="index_sdg", y="department",
           color="period",
           labels={"index_sdg": "IDS (population-weighted)"},
           title="Sustainable Development Index by department")

## Where progress reached most people

In [ ]:
df = geo.load("sdg")
wide = df.pivot(index="geo_id", columns="year", values=["index_sdg", "population"])
gain = wide[("index_sdg", 2024)] - wide[("index_sdg", 2012)]
out = (gain * wide[("population", 2024)]).rename("points").reset_index()
names = geo.units("map343").set_index("geo_id")["name"]
out["name"] = out["geo_id"].map(names)
px.bar(out.nlargest(12, "points"), x="points", y="name",
       labels={"points": "Gain in the IDS × population 2024", "name": ""},
       title="Units with the largest population-weighted progress")

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- Lykke E. Andersen and Fabiana Argandoña (SDSN Bolivia, Universidad Privada Boliviana): [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)
- SDSN Bolivia, from the Ministerio de Planificación del Desarrollo, the Instituto Nacional de Estadística and the Viceministerio de Autonomías: [Creative Commons Attribution 4.0 International License (licensed by SDSN Bolivia)](https://creativecommons.org/licenses/by/4.0/)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La Paz: Universidad Privada Boliviana, SDSN Bolivia.
2. SDSN Bolivia (2025). Límites Municipales Bolivia 2025 [Shapefile], with the population of the 2012 and 2024 Population and Housing Censuses (INE). https://sdsnbolivia.org/datos-espaciales/
3. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
